# Vanilla, SFT, SFT + GRPO and GRPO-only graph navigation

This notebook uses **real graph tools, assistant-only supervised fine-tuning, and Hugging Face TRL's actual GRPOTrainer**. SFT learns tool-call sequences from verified training-only demonstrations. GRPO then samples tool-call tokens and optimizes correctness plus a one-time verified retrieval bonus. Real observations enter subsequent generations; environment tokens never contribute to either training objective.

**Saved development run:** CPU preflight only. GPU-dependent results are absent unless this notebook is executed with the selected CUDA stack. No pretrained-model rollout, successful optimization step, T4 memory measurement, or adapter reload is claimed from the CPU outputs.

**Kaggle quick start**
1. Create a Python notebook, enable **one T4** GPU and internet for GitHub, package and model downloads.
2. The configuration cell clones `https://github.com/AlirezaAK2000/kg-navigation-rlvr.git` into the **current working directory** (normally `/kaggle/working/`), so code is available as `/kaggle/working/src/kg_rlvr/`, with **no `kg-navigation-rlvr/` subfolder**. The repository includes the saved dataset; a separately attached dataset is optional.
3. Configure the paths and switches below. For a fresh runtime with different packages, set `INSTALL_DEPENDENCIES=True` and run the setup cell. Restart the session if instructed, then set it back to `False` and run all cells.
4. The default GPU run evaluates the base model, performs **one SFT epoch on 200 reserved demonstrations**, evaluates SFT-only performance, and attempts **eight genuine GRPO smoke steps** on separate questions. GRPO-only then runs from the saved vanilla initialization with the same RL budget. Full 30-step training for each RL variant is explicit opt-in: `RUN_FULL_TRAINING=True`.
5. Keep outputs under `/kaggle/working/`. Save a Kaggle notebook version **with outputs** or download the ZIP to retain files after the session ends.

**Contents:** [1 Setup](#setup) · [2 Hardware](#hardware) · [3 Data](#data) · [4 Tools](#tools) · [5 Model](#model) · [6 Baseline](#baseline) · [6b SFT](#sft) · [7 GRPO](#grpo) · [8 Smoke](#smoke) · [9 Full training](#full) · [10 Metrics](#metrics) · [11 Evaluation](#evaluation) · [12 Export](#export) · [13 GRPO only](#grpo-only) · [14 Comparison](#comparison) · [15 ZIP](#zip-heading)

All four policies use the same fixed test question IDs. Default order: vanilla test evaluation → verified SFT demonstrations → SFT-only evaluation → GRPO smoke/full → final evaluation. SFT and GRPO use disjoint training query instances, including all paraphrases; test stays outside both training objectives. SFT + GRPO continues the verified SFT adapter. GRPO only restores the saved initial adapter. Both RL variants use the same GRPO pool and budgets; each saves its results before the next variant.


<a id="setup"></a>
## 1. Environment setup and pinned dependencies

**Get the project from GitHub directly into the current folder.** Leave `CLONE_REPOSITORY=True` and `PROJECT_ROOT_OVERRIDE=None` for the normal Kaggle workflow. With an empty working directory this is equivalent to:

```bash
git clone --depth 1 --branch master https://github.com/AlirezaAK2000/kg-navigation-rlvr.git .
```

Kaggle can place files in `/kaggle/working/` before this notebook runs. The setup below also handles that case: it clones to a temporary location and copies the checkout contents, including `.git`, into `Path.cwd()`. It refuses conflicting files before copying and preserves unrelated working files. No repository-named subfolder is created in the working directory. An existing project is reused on reruns without pulling or overwriting it. `PROJECT_ROOT` then resolves to the current folder, and `DATA_ROOT` defaults to its `data/` directory.

If you already mounted the project as a Kaggle input dataset, set `CLONE_REPOSITORY=False` and supply `PROJECT_ROOT_OVERRIDE` instead. For separately attached graph/QA data, set `DATA_ROOT_OVERRIDE` to that mount. Internet must be enabled for the GitHub clone.

All user choices are collected here. `DATA_ROOT` is not tied to a Kaggle slug. Inputs are read-only; checkpoints, logs, adapters, and figures are written elsewhere. CPU execution runs the non-training checks and explicitly skips model phases.

The selected stack is Torch **2.6.0/cu124**, Transformers **5.3.0**, TRL **1.8.0**, PEFT **0.18.1**, Accelerate **1.10.1**, bitsandbytes **0.49.2**, datasets **4.8.3**. It was import/API checked on CPU; the CUDA/QLoRA combination still requires the smoke test below. NumPy **1.26.4**, Pandas **2.2.3** and PyArrow **21.0.0** are pinned together to avoid binary-version mismatches. Target Python is 3.11/3.12.

Sources: [released TRL documentation](https://huggingface.co/docs/trl/v1.8.0/en/grpo_trainer), [TRL 1.8.0 implementation](https://github.com/huggingface/trl/blob/v1.8.0/trl/trainer/grpo_trainer.py), [Transformers 5.3.0](https://huggingface.co/docs/transformers/v5.3.0/en/index), [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization), [bnb 0.49.2 supported builds](https://huggingface.co/docs/bitsandbytes/v0.49.2/en/installation), [official Torch CUDA wheels](https://pytorch.org/get-started/previous-versions/).

No vLLM, FlashAttention, reward model, LLM judge, or sentencepiece installation is needed. Qwen3 uses its BPE tokenizer and native tool-call chat template. Setup optionally removes unused audio/vision packages from the **fresh Kaggle session** to avoid their compiled extensions conflicting with the pinned Torch version. If a changed library was already imported, restart before continuing; package installation is not a training smoke test.
The graph backend requires **RDFLib 7.6.0** and **pyparsing 3.2.3**, included in `requirements-grpo.txt`. The saved graph is `data/graph/graph.ttl` (OWL schema plus labeled individuals and asserted edges); open it in Protégé for inspection. Older JSON/TSV graph mounts remain readable.

**Fresh Kaggle images need installation first.** `INSTALL_DEPENDENCIES=False` skips pip; it does not mean that the image already has these pins. If section 1 reports missing/different packages (or an older copy stops in section 5), set it to `True`, rerun configuration and setup, wait for installation to finish, then restart the session/kernel. Set it back to `False` and run from the beginning. TRL and bitsandbytes must be installed even for the vanilla baseline because this notebook shares its QLoRA loading stack. Do not comment out version checks to continue with an untested API mix.


In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata, tempfile, shutil

# GITHUB: clone into Path.cwd() on Kaggle; never create a repository-named subfolder.
CLONE_REPOSITORY = True
REPOSITORY_URL = "https://github.com/AlirezaAK2000/kg-navigation-rlvr.git"
REPOSITORY_BRANCH = "master"

# PATHS: None uses the cloned working directory and its saved data/.
PROJECT_ROOT_OVERRIDE = None  # alternate: Path("/kaggle/input/my-project/graph_rlvr")
DATA_ROOT_OVERRIDE = None     # e.g. Path("/kaggle/input/my-kg-dataset/data")
WORK_ROOT_OVERRIDE = None     # None -> /kaggle/working on Kaggle; local artifacts/grpo otherwise

# EXECUTION: a GPU smoke run is enabled; full training is opt-in.
INSTALL_DEPENDENCIES = False
REMOVE_UNUSED_MULTIMODAL_PACKAGES = True
RUN_GPU = True
SFT_EXAMPLES = 200           # required supervised warm start; reserved separately from GRPO
SFT_EPOCHS = 1
SFT_LEARNING_RATE = 2e-5
SFT_GRADIENT_ACCUMULATION_STEPS = 8
SHOW_PROGRESS = True          # questions, tool turns, sampled trajectories and optimizer steps
INSPECT_EXAMPLES = 3           # display stored episodes, without extra inference
INSPECT_OFFSET = 0             # choose another consecutive slice; same indices for vanilla/SFT/SFT+GRPO test outputs
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = False
RUN_GRPO_ONLY = True          # independent RL baseline, restored from the saved vanilla initialization
RUN_NAME = "kg_four_variants_seed41"
SELECTED_GPU = "0"  # expose one device BEFORE importing torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MODEL_REVISION_OVERRIDE = None  # immutable known revision selected below
TRAIN_HOPS = [1, 2, 3]
TRAIN_HOP_EXPOSURE = {1: 0.2, 2: 0.4, 3: 0.4}  # proportions of actual sampled question groups
SINGLE_ANSWER_ONLY = True     # excludes multi-answer queries from every model phase
EVAL_HOPS = [1, 2, 3, 4]
MAX_TRAIN_EXAMPLES = 1000       # GRPO pool cap AFTER reserving the separate SFT examples
TEST_EVAL_SIZE = 30           # same fixed test subset for all four variants; None uses all unique test instances
RELOAD_EVAL_SIZE = 3
SEED = 41

NUM_GENERATIONS = 8
LORA_R, LORA_ALPHA = 8, 16
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-6
MAX_TOOL_CALLS = 12           # four discovery/traversal pairs + submission need 9 calls
MAX_TURN_TOKENS = 128
MAX_COMPLETION_TOKENS = 1536    # all model + observation tokens after the initial prompt
MAX_PROMPT_TOKENS = 1024
MAX_CONTEXT_TOKENS = 3072
MAX_NEIGHBORS_PER_RESPONSE = 16
MAX_OBSERVATION_CHARS = 4000
SMOKE_TEST_STEPS = 8         # 8 groups / 64 sampled episodes with the default G=8, accumulation=8
INITIAL_TRAINING_STEPS = 30
SAVE_STEPS = 10
EVALUATE_EVERY_STEPS = 1        # test monitoring after every optimizer step in smoke/full; 0 disables
PERIODIC_EVAL_SIZE = 6
GRPO_ONLY_RESUME_FROM_CHECKPOINT = None
RESUME_FROM_CHECKPOINT = None   # full checkpoint-* folder, not an adapter-only export
MAKE_ZIP = True

def clone_into_current_directory(repository_url, branch):
    target = Path.cwd().resolve()
    if (target / "src/kg_rlvr/graph_index.py").is_file():
        print(f"Project already present in {target}; skipping clone (no automatic pull).")
        return
    if (target / ".git").exists():
        raise FileExistsError("Current folder belongs to another Git checkout. Use a fresh working folder or set PROJECT_ROOT_OVERRIDE.")
    if not any(target.iterdir()):
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repository_url, "."], cwd=target)
    else:
        # A live Kaggle notebook/output can make the current directory nonempty.
        # Stage outside it, check every destination, then copy checkout contents.
        with tempfile.TemporaryDirectory(prefix="kg_navigation_checkout_") as temporary:
            checkout = Path(temporary) / "checkout"
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repository_url, str(checkout)])
            conflicts = []
            for source in checkout.rglob("*"):
                destination = target / source.relative_to(checkout)
                if destination.is_symlink() or (destination.exists() and (
                    source.is_dir() != destination.is_dir()
                    or (source.is_file() and source.read_bytes() != destination.read_bytes())
                )):
                    conflicts.append(str(source.relative_to(checkout)))
            if conflicts:
                raise FileExistsError("Clone would overwrite different existing files; nothing copied. Conflicts: " + ", ".join(sorted(conflicts)[:10]))
            shutil.copytree(checkout, target, dirs_exist_ok=True)
    if not (target / "src/kg_rlvr/graph_index.py").is_file():
        raise FileNotFoundError("Cloned branch is missing src/kg_rlvr/graph_index.py")
    print(f"Repository ready directly in {target}; no repository-name subfolder created.")

IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
if IS_KAGGLE and CLONE_REPOSITORY and PROJECT_ROOT_OVERRIDE is None:
    clone_into_current_directory(REPOSITORY_URL, REPOSITORY_BRANCH)

def find_project():
    if PROJECT_ROOT_OVERRIDE:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src/kg_rlvr/graph_index.py").exists():
            return parent
    candidates = list(Path("/kaggle/input").glob("*/src/kg_rlvr/graph_index.py"))
    candidates += list(Path("/kaggle/input").glob("*/*/src/kg_rlvr/graph_index.py"))
    if len(candidates) == 1:
        return candidates[0].parents[2]
    raise FileNotFoundError("Enable the Kaggle GitHub clone or set PROJECT_ROOT_OVERRIDE to the repository containing src/kg_rlvr.")

PROJECT_ROOT = find_project()
DATA_ROOT = Path(DATA_ROOT_OVERRIDE or PROJECT_ROOT / "data").expanduser().resolve()
WORK_ROOT = Path(WORK_ROOT_OVERRIDE or ("/kaggle/working" if IS_KAGGLE else PROJECT_ROOT / "artifacts/grpo")).expanduser().resolve()
if WORK_ROOT == DATA_ROOT or DATA_ROOT in WORK_ROOT.parents:
    raise ValueError("Working outputs must be outside the source dataset")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be a simple directory name")
if GRADIENT_ACCUMULATION_STEPS < NUM_GENERATIONS or GRADIENT_ACCUMULATION_STEPS % NUM_GENERATIONS:
    raise ValueError("Microbatch 1 requires accumulation divisible by NUM_GENERATIONS")
if NUM_GENERATIONS < 2:
    raise ValueError("GRPO requires at least two generations per prompt")
if set(TRAIN_HOPS) - {1,2,3,4} or not TRAIN_HOPS or set(EVAL_HOPS) - {1,2,3,4} or not EVAL_HOPS:
    raise ValueError("Choose hop categories from 1, 2, 3, 4")
if set(TRAIN_HOP_EXPOSURE) != set(TRAIN_HOPS):
    raise ValueError("TRAIN_HOP_EXPOSURE must specify every selected TRAIN_HOPS category")
if SFT_EXAMPLES < 1:
    raise ValueError("SFT requires at least one reserved training example")
if TEST_EVAL_SIZE is not None and TEST_EVAL_SIZE < 1:
    raise ValueError("TEST_EVAL_SIZE must be positive or None for the full unique test pool")
if SFT_EPOCHS <= 0 or SFT_LEARNING_RATE <= 0 or SFT_GRADIENT_ACCUMULATION_STEPS < 1:
    raise ValueError("SFT epochs, learning rate and accumulation must be positive")
if EVALUATE_EVERY_STEPS < 0 or PERIODIC_EVAL_SIZE < 1:
    raise ValueError("Evaluation interval must be nonnegative and periodic evaluation size positive")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CUDA_VISIBLE_DEVICES"] = SELECTED_GPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "kg_grpo_matplotlib"))
os.environ.setdefault("HF_HOME", str(WORK_ROOT / "hf_cache"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT = WORK_ROOT / "runs" / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints" / RUN_NAME
VARIANT_DIRS = {name: RUN_ROOT / "variants" / name for name in ["vanilla", "sft", "sft_grpo", "grpo_only"]}
EVALUATION_DIR = VARIANT_DIRS["sft_grpo"] / "evaluation"
ADAPTER_DIR = VARIANT_DIRS["sft_grpo"] / "adapter"
INITIAL_ADAPTER_DIR = VARIANT_DIRS["vanilla"] / "initial_adapter"
SFT_GRPO_CHECKPOINT_ROOT = CHECKPOINT_ROOT / "sft_grpo"
GRPO_ONLY_CHECKPOINT_ROOT = CHECKPOINT_ROOT / "grpo_only"
print(f"Project: {PROJECT_ROOT}\nData: {DATA_ROOT}\nWorking outputs: {WORK_ROOT}\nSingle GPU selection: {SELECTED_GPU}")

Project: /Users/alirezaak/Documents/research/graph_rlvr
Data: /Users/alirezaak/Documents/research/graph_rlvr/data
Working outputs: /Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo
Single GPU selection: 0


In [2]:
if INSTALL_DEPENDENCIES:
    if not IS_KAGGLE:
        raise RuntimeError("Automatic CUDA installation is scoped to a fresh Kaggle session. For local CPU checks use requirements.txt; inspect requirements-grpo.txt separately.")
    loaded = {name for name in ["torch", "numpy", "pandas", "transformers", "trl", "peft", "accelerate", "datasets", "pyarrow", "rdflib", "pyparsing"] if name in sys.modules}
    if REMOVE_UNUSED_MULTIMODAL_PACKAGES:
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "torchcodec", "librosa"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-r", str(PROJECT_ROOT / "requirements-grpo.txt")])
    if loaded:
        raise RuntimeError(f"Setup changed libraries already imported ({sorted(loaded)}). Restart the Kaggle session, set INSTALL_DEPENDENCIES=False, then Run All. Do not continue with mixed in-memory versions.")
# Check package metadata before importing the graph/ML stack. A fresh Kaggle
# image can lack TRL, bitsandbytes or RDFLib entirely; fail here, not in section 5.
if IS_KAGGLE and RUN_GPU:
    expected_packages = {}
    for line in (PROJECT_ROOT / "requirements-grpo.txt").read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            name, pinned = line.split("==", 1)
            expected_packages[name] = pinned.split(";", 1)[0].strip()
    setup_mismatches = {}
    for name, expected in expected_packages.items():
        try:
            actual = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            actual = None
        if (actual or "").split("+", 1)[0] != expected:
            setup_mismatches[name] = {"expected": expected, "actual": actual}
    if setup_mismatches:
        raise RuntimeError(
            "Kaggle dependencies are not ready. In section 1, set INSTALL_DEPENDENCIES=True "
            "and rerun the configuration and setup cells. Wait for pip to finish, then "
            "restart the Kaggle session/kernel, set INSTALL_DEPENDENCIES=False, and run "
            "from the beginning. For vanilla-baseline-only evaluation: "
            "Stop after section 6 to inspect vanilla before SFT. "
            "Do not bypass the pins. Package differences: " + json.dumps(setup_mismatches))

from kg_rlvr.training.grpo import dependency_versions, MODEL_REVISIONS, PINS
MODEL_REVISION = MODEL_REVISION_OVERRIDE or MODEL_REVISIONS.get(MODEL_NAME)
if not MODEL_REVISION:
    raise ValueError("Supply an explicit immutable revision for a different model")
print("Requested pins:", json.dumps(PINS, indent=2))
print("Installed:", json.dumps(dependency_versions(), indent=2))
print("Selected model/revision:", MODEL_NAME, MODEL_REVISION)

Requested pins: {
  "torch": "2.6.0",
  "transformers": "5.3.0",
  "trl": "1.8.0",
  "peft": "0.18.1",
  "accelerate": "1.10.1",
  "bitsandbytes": "0.49.2",
  "datasets": "4.8.3",
  "jmespath": "1.0.1",
  "numpy": "1.26.4",
  "pandas": "2.2.3",
  "pyarrow": "21.0.0",
  "rdflib": "7.6.0",
  "pyparsing": "3.2.3"
}
Installed: {
  "torch": "2.14.0",
  "transformers": "4.48.1",
  "trl": null,
  "peft": null,
  "accelerate": "1.10.0",
  "bitsandbytes": null,
  "datasets": "4.4.1",
  "jmespath": "0.10.0",
  "numpy": "1.26.4",
  "pandas": "2.3.2",
  "pyarrow": "22.0.0",
  "rdflib": "7.6.0",
  "pyparsing": "3.2.3",
  "tokenizers": "0.21.0",
  "huggingface-hub": "0.36.2",
  "matplotlib": "3.7.2",
  "psutil": "5.9.0",
  "tqdm": "4.67.1"
}
Selected model/revision: Qwen/Qwen3-1.7B 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e


<a id="hardware"></a>
## 2. GPU, CUDA, VRAM, and system memory

Only the selected GPU is visible to Torch; the `nvidia-smi` inventory can still show all physical devices. Two T4s do not provide a pooled 32 GB device for this notebook. The T4 path uses FP16 computation because it lacks native BF16 support; the runtime capability check selects BF16 only when supported. Training uses microbatch 1, gradient checkpointing, frozen NF4 base weights, and trainable LoRA parameters.

In [3]:
import gc, math, random, platform, time
import warnings
with warnings.catch_warnings(record=True) as dependency_notices:
    warnings.simplefilter("always", UserWarning)
    import pandas as pd
import torch
import psutil
from IPython.display import display, Markdown, HTML
from kg_rlvr.training.grpo import memory_snapshot, fingerprint_inputs, native_bf16_supported
from kg_rlvr.training.environment import EpisodeLimits, load_tasks, public_prompt, GraphEnvironment, TOOL_SCHEMAS, REWARD_SPEC

try:
    inventory = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"], text=True, capture_output=True, check=False)
    print(inventory.stdout.strip() or "nvidia-smi inventory unavailable")
except FileNotFoundError:
    print("nvidia-smi unavailable on this machine")
HAS_CUDA = torch.cuda.is_available()
RUN_MODEL = RUN_GPU and HAS_CUDA
hardware = {"python": platform.python_version(), "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
            "visible_cuda_devices": torch.cuda.device_count(), "system_available_gib": psutil.virtual_memory().available / 2**30}
if HAS_CUDA:
    properties = torch.cuda.get_device_properties(0)
    hardware.update(gpu_name=properties.name, gpu_vram_gib=properties.total_memory / 2**30,
                    gpu_capability=torch.cuda.get_device_capability(0), bf16_native_supported=native_bf16_supported())
display(pd.Series(hardware, name="runtime hardware").to_frame())
if dependency_notices:
    display(pd.DataFrame({"environment notice": [str(w.message) for w in dependency_notices]}))
MEMORY = [memory_snapshot("before_model")]
if not RUN_MODEL:
    display(Markdown("**CPU preflight mode:** model downloads, inference, training, GPU memory measurements, and adapter save/reload are skipped. Subsequent skipped sections do not count as successful GPU tests."))
LIMITS = EpisodeLimits(MAX_TOOL_CALLS, MAX_NEIGHBORS_PER_RESPONSE, MAX_OBSERVATION_CHARS,
                      MAX_TURN_TOKENS, MAX_COMPLETION_TOKENS, MAX_PROMPT_TOKENS, MAX_CONTEXT_TOKENS)
random.seed(SEED)
torch.manual_seed(SEED)

nvidia-smi unavailable on this machine


,runtime hardware
python,3.11.5
torch,2.14.0
cuda_runtime,None
visible_cuda_devices,0
system_available_gib,0.727753


,environment notice
0,Pandas requires version '1.3.6' or newer of 'b...


**CPU preflight mode:** model downloads, inference, training, GPU memory measurements, and adapter save/reload are skipped. Subsequent skipped sections do not count as successful GPU tests.

<a id="data"></a>
## 3. Single-answer dataset selection and model/verifier separation

Validate the complete source dataset, then remove questions whose complete gold answer set has more than one entity from the active experiment. Use reviewed singular templates for retained questions, preserving original IDs and train/test assignments. Filtering applies before deduplication and limits, and identically to baseline, smoke/full training, periodic test monitoring, final test evaluation, and adapter reload. Source datasets remain intact for reproducibility; the active filtered records are exported under the run's `selected_dataset/rl/` directory.

The selection report shows source/eligible/removed question counts and eligible unique instances by split and hop. Training uses **20% one-hop, 40% two-hop, 40% three-hop question groups**. Its schedule enforces exposure during actual training; the selected pool proportions alone do not determine exposure. Evaluation remains balanced across its selected hop categories.

Reserve `SFT_EXAMPLES=200` distinct instances (40/80/80 by hop), then apply `MAX_TRAIN_EXAMPLES` to the remaining GRPO pool. Every stage is checked for overlapping question IDs, complete query instances and public prompts. The default full source gives 200 SFT and 733 GRPO instances. The graph itself is shared; this is question-instance separation, not a disjoint-entity experiment. Partitions and verified reference demonstrations are saved before model training.


In [4]:
from collections import Counter
from dataclasses import asdict
from kg_rlvr.training.sampling import exposure_schedule, partition_training_tasks, assert_disjoint_tasks
from kg_rlvr.training.sft import build_demonstration
import hashlib
INPUT_SHA256 = fingerprint_inputs(DATA_ROOT)
graph, store, dataset_validation = load_tasks(DATA_ROOT, max_tool_calls=MAX_TOOL_CALLS, validate=True, single_answer_only=SINGLE_ANSWER_ONLY)
# Reference tools validate each SFT candidate before it is reserved. No model is run.
sft_demo_cache = {}
def check_sft_candidate(qid):
    sft_demo_cache[qid] = build_demonstration(store, qid, LIMITS)
sft_ids, train_ids, partition_report = partition_training_tasks(store, TRAIN_HOPS,
    sft_count=SFT_EXAMPLES, hop_weights=TRAIN_HOP_EXPOSURE, max_grpo_examples=MAX_TRAIN_EXAMPLES,
    seed=SEED, sft_candidate_check=check_sft_candidate)
SFT_DEMONSTRATIONS = [sft_demo_cache[qid] for qid in sft_ids]
test_ids = store.select("test", EVAL_HOPS, TEST_EVAL_SIZE, SEED + 1, unique_instances=True, balanced=True)
partition_report["all_stage_check"] = assert_disjoint_tasks(store, {
    "sft": sft_ids, "grpo": train_ids, "test": test_ids})
partition_report["sft_ids"] = sft_ids
partition_report["grpo_ids"] = train_ids
STAGE_PARTITION_SHA256 = hashlib.sha256(json.dumps(partition_report, sort_keys=True).encode()).hexdigest()
(RUN_ROOT / "stage_partition.json").write_text(json.dumps(partition_report, indent=2) + "\n")
for stage, ids in [("sft", sft_ids), ("grpo", train_ids)]:
    stage_dir = RUN_ROOT / "training_partitions"
    stage_dir.mkdir(parents=True, exist_ok=True)
    (stage_dir / f"{stage}.jsonl").write_text("".join(json.dumps(store.record(qid)) + "\n" for qid in ids))
SFT_RUN_DIR = VARIANT_DIRS["sft"]
SFT_RUN_DIR.mkdir(parents=True, exist_ok=True)
(SFT_RUN_DIR / "demonstrations.jsonl").write_text("".join(json.dumps(demo) + "\n" for demo in SFT_DEMONSTRATIONS))
display(pd.Series(partition_report["all_stage_check"], name="verified stage separation").to_frame())
display(pd.DataFrame([{"subset": label, "questions": len(ids), "hops": dict(Counter(store.record(qid)["hop_count"] for qid in ids))}
                      for label, ids in [("SFT reserved", sft_ids), ("GRPO train", train_ids), ("fixed test (all four variants)", test_ids)]]))
print(f"Graph: {len(graph.entities)} entities, {len(graph.triples)} stored triples; dataset valid: {dataset_validation['valid']}")
print("Selected train rows are real existing query instances; a configured maximum larger than the available pool is reported, not filled with invented rows.")
display(Markdown("**Example model-visible prompt (no gold program or answer metadata):**"))
display(pd.DataFrame(store.prompt(train_ids[0])))
private_probe = store.record(train_ids[0])
private_probe.update(answer_ids=["VERIFIER_PRIVATE_SENTINEL"], gold_entity_paths=[["VERIFIER_PRIVATE_SENTINEL"]], relation_path=["VERIFIER_PRIVATE_SENTINEL"])
assert public_prompt(graph, private_probe, MAX_TOOL_CALLS, SINGLE_ANSWER_ONLY) == store.prompt(train_ids[0])
assert "VERIFIER_PRIVATE_SENTINEL" not in json.dumps(store.prompt(train_ids[0]))
fixed_selection = {"sft_ids": sft_ids, "train_ids": train_ids, "test_ids": test_ids}
(RUN_ROOT / "fixed_task_ids.json").write_text(json.dumps(fixed_selection, indent=2) + "\n")
# Filtering precedes deduplication and example caps. Preserve original split/IDs.
display(pd.DataFrame(store.selection_statistics))
(RUN_ROOT / "question_selection.json").write_text(json.dumps(store.selection_statistics, indent=2) + "\n")
for split in ["train", "test"]:
    eligible = store.select(split, [1, 2, 3, 4], seed=SEED, unique_instances=False)
    if SINGLE_ANSWER_ONLY:
        assert all(len(store.record(qid)["answer_ids"]) == 1 for qid in eligible)
    export_dir = RUN_ROOT / "selected_dataset" / "rl"
    export_dir.mkdir(parents=True, exist_ok=True)
    (export_dir / f"{split}.jsonl").write_text("".join(json.dumps(store.record(qid)) + "\n" for qid in eligible))
exposure_preview = []
for phase, steps in [("smoke", SMOKE_TEST_STEPS), ("full", INITIAL_TRAINING_STEPS)]:
    planned = exposure_schedule(store, train_ids, steps * (GRADIENT_ACCUMULATION_STEPS // NUM_GENERATIONS), TRAIN_HOP_EXPOSURE, SEED)
    exposure_preview.append({"phase": phase, "question_groups": len(planned),
        "groups_by_hop": dict(Counter(store.record(qid)["hop_count"] for qid in planned)),
        "sampled_trajectories": len(planned) * NUM_GENERATIONS})
display(pd.DataFrame(exposure_preview))
(RUN_ROOT / "training_exposure_preview.json").write_text(json.dumps(exposure_preview, indent=2) + "\n")
print("20/40/40 exposure means 1/2/2 question groups in every five groups; partial blocks are rounded by interleaving.")

from kg_rlvr.training.artifacts import evaluate_variant, save_json, save_comparison, read_rows, collect_training_logs
RUN_METADATA = {"model_name": MODEL_NAME, "revision": MODEL_REVISION, "hardware": hardware,
    "versions": dependency_versions(), "source_sha256": INPUT_SHA256, "limits": asdict(LIMITS),
    "evaluation_split": "test", "evaluation_task_ids": test_ids, "stage_partition_sha256": STAGE_PARTITION_SHA256,
    "variants": ["vanilla", "sft", "sft_grpo", "grpo_only"], "grpo_only_enabled": RUN_GRPO_ONLY}
save_json(RUN_ROOT / "run_environment.json", RUN_METADATA)


,verified stage separation
disjoint,True
checked,"[question_ids, query_instances, public_prompts]"
counts,"{'sft': 200, 'grpo': 733, 'test': 30}"


,subset,questions,hops
0,SFT reserved,200,"{2: 80, 1: 40, 3: 80}"
1,GRPO train,733,"{1: 277, 2: 229, 3: 227}"
2,fixed test (all four variants),30,"{1: 8, 2: 8, 3: 7, 4: 7}"


Graph: 2000 entities, 5000 stored triples; dataset valid: True
Selected train rows are real existing query instances; a configured maximum larger than the available pool is reported, not filled with invented rows.


**Example model-visible prompt (no gold program or answer metadata):**

,role,content
0,system,Navigate the fictional research graph to answe...
1,user,Question: Which publication was produced by Pr...


,split,hop_count,source_questions,eligible_questions,removed_questions,eligible_unique_instances
0,test,1,267,249,18,125
1,test,2,267,249,18,125
2,test,3,267,249,18,125
3,test,4,266,236,30,118
4,train,1,667,633,34,317
5,train,2,667,617,50,309
6,train,3,667,613,54,307
7,train,4,666,586,80,293


,phase,question_groups,groups_by_hop,sampled_trajectories
0,smoke,8,"{2: 3, 3: 3, 1: 2}",64
1,full,30,"{2: 12, 3: 12, 1: 6}",240


20/40/40 exposure means 1/2/2 question groups in every five groups; partial blocks are rounded by interleaving.


<a id="tools"></a>
## 4. Stateful real graph environment

Every rollout creates a new GraphEnvironment and discovery-gated NavigationSession. Tools expose only discovered entity IDs, readable labels, types and relation operations. Invalid calls consume the same call budget as valid calls. Inverse traversal follows the existing ontology; one-to-many responses are complete. A response exceeding an explicit observation limit returns an error rather than silently hiding neighbors. The graph is shared read-only; discoveries, submissions, call counters and logs are isolated.

Submission ends the episode immediately. Its acknowledgment never reveals correctness. Plain-text final answers are not submissions. Malformed calls and token/context truncation are recorded, never repaired into fictional model actions.

The following **deterministic tool contract check is not a model trajectory or baseline**. It uses an existing reference path only to test graph operations and binary verification; gold programs are never supplied to a model.

In [5]:
fixture_id = store.select("train", [min(TRAIN_HOPS)], count=1, seed=SEED)[0]
fixture = store.record(fixture_id)
fixture_env = GraphEnvironment(graph, fixture["start_entity"], LIMITS)
fixture_path = fixture["gold_entity_paths"][0]
for head, relation, tail in zip(fixture_path, fixture["relation_path"], fixture_path[1:]):
    observation = fixture_env.get_neighbors(head, relation)
    assert observation["ok"] and tail in [entity["id"] for entity in observation["entities"]]
ack = fixture_env.submit_answer(fixture_path[-1])
assert fixture_env.done and fixture_env.termination_reason == "submitted"
assert "reward" not in ack and "correct" not in ack
assert store.reward(fixture_id, fixture_env.submitted_answer) == 1.0
assert store.reward(fixture_id, None) == 0.0
assert store.reward(fixture_id, fixture["start_entity"]) == 0.0
verified_fixture = store.verify_episode(fixture_id, fixture_env.submitted_answer, fixture_env.trajectory)
assert verified_fixture == {"answer_correct": 1.0, "gold_entity_retrieved": True, "retrieval_reward": 0.2, "training_reward": 1.2}
assert store.verify_episode(fixture_id, None, fixture_env.trajectory)["training_reward"] == 0.2
assert store.verify_episode(fixture_id, None, [])["training_reward"] == 0.0
isolation_probe = GraphEnvironment(graph, fixture["start_entity"], LIMITS)
assert isolation_probe.calls == 0 and isolation_probe.submitted_answer is None
display(pd.DataFrame(fixture_env.trajectory))
print("Non-model tool contract checks passed. Schema tools:", [s["function"]["name"] for s in TOOL_SCHEMAS])

,index,name,arguments,observation
0,1,get_neighbors,"{'entity_id': 'researcher_000562', 'relation_i...","{'ok': True, 'entity_id': 'researcher_000562',..."
1,2,submit_answer,{'entity_id': 'project_000257'},"{'ok': True, 'submitted': 'project_000257', 'e..."


Non-model tool contract checks passed. Schema tools: ['get_relations', 'get_neighbors', 'submit_answer']


<a id="model"></a>
## 5. Qwen3 initialization with NF4 QLoRA

The exact Hub revision is pinned. All seven Qwen attention/MLP projection suffixes are verified against the loaded architecture before applying rank-8, alpha-16 LoRA with dropout 0. Only LoRA parameters may require gradients. The base model is loaded once on one GPU, with double-quantized NF4, SDPA attention, and non-reentrant gradient checkpointing. Training caches are disabled. Generation uses one bounded KV cache at a time; rollouts are sampled sequentially rather than holding four inference batches in memory.

**Memory fallback, in order:** lower `MAX_TURN_TOKENS`/`MAX_COMPLETION_TOKENS`/context limits; lower tool interactions if the tasks still fit; reduce optional evaluation frequency and clear old trainer/optimizer objects; reduce `NUM_GENERATIONS` from 4 to 2 **with matching accumulation divisibility**. If still necessary, explicitly set `MODEL_NAME="Qwen/Qwen3-0.6B"` and rerun from model initialization. The notebook never switches model silently. Lowering the budget changes the task and should be reflected in a fresh baseline. With eight calls, a four-hop agent cannot list relations separately at every stage plus submit (that costs nine); it can reuse relation knowledge or you can explicitly increase the budget for both evaluations.
Generation explicitly uses the same native CUDA AMP dtype as the trainer, including the baseline before Accelerate wraps the model. KV caching is enabled only for bounded, gradient-free generation; training keeps `model.config.use_cache=False`.


In [6]:
from kg_rlvr.training.grpo import load_qlora, assert_training_stack, export_adapter, lora_fingerprint
from kg_rlvr.training.rollouts import QwenChatCodec, progress_bar
model, tokenizer, COMPUTE_DTYPE = None, None, None
if RUN_MODEL:
    assert_training_stack(require_cuda=True)
    try:
        model, tokenizer, COMPUTE_DTYPE = load_qlora(MODEL_NAME, MODEL_REVISION, LORA_R, LORA_ALPHA, show_progress=SHOW_PROGRESS)
    except torch.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("Model load ran out of memory. Apply the documented smaller-budget/model fallback explicitly; no replacement model has been loaded.")
    INITIAL_LORA_SHA256 = lora_fingerprint(model)
    export_adapter(model, tokenizer, INITIAL_ADAPTER_DIR, {**RUN_METADATA, "kind": "untrained_initial_adapter",
        "initial_lora_sha256": INITIAL_LORA_SHA256})
    codec = QwenChatCodec(tokenizer)
    selected_prompt_ids = sorted(set(sft_ids + train_ids + test_ids))
    prompt_lengths = []
    with progress_bar(len(selected_prompt_ids), "Checking model prompts", enabled=SHOW_PROGRESS) as progress:
        for qid in selected_prompt_ids:
            prompt_lengths.append(len(codec.prompt_ids(store.prompt(qid))))
            progress.update(1)
    if max(prompt_lengths) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Largest selected prompt has {max(prompt_lengths)} tokens > configured {MAX_PROMPT_TOKENS}")
    format_probe = '<tool_call>\n{"name":"get_relations","arguments":{"entity_id":"' + fixture["start_entity"] + '"}}\n</tool_call><|im_end|>'
    from kg_rlvr.training.environment import parse_qwen_action
    assert parse_qwen_action(format_probe)[0] == "get_relations"
    feedback = codec.decode(codec.feedback_ids("get_relations", {"ok": True, "relations": ["works_at"]}))
    assert "<tool_response>" in feedback
    print("Model", MODEL_NAME, "revision", MODEL_REVISION, "dtype", COMPUTE_DTYPE, "max public prompt tokens", max(prompt_lengths))
    MEMORY.append(memory_snapshot("after_qlora_load"))
else:
    print("Qwen weights and NF4 initialization not run: no enabled CUDA model runtime.")

Qwen weights and NF4 initialization not run: no enabled CUDA model runtime.


<a id="baseline"></a>
## 6. Vanilla test evaluation before training

Disable the adapter to evaluate the vanilla base checkpoint. All four policies use the **same ordered `test_ids`**, graph tools, discovery rules, token limits and call budget, with greedy decoding and no gold guidance. The default `TEST_EVAL_SIZE=30` selects a balanced diagnostic subset from test; set it to `None` to evaluate every eligible unique test instance. There is no validation split.

To inspect vanilla before any training, execute sections 1–6 and stop after the inspection cell. SFT runs next; GRPO must continue its verified adapter. CPU preflight skips all model phases.

With `SHOW_PROGRESS=True`, each model evaluation shows completed questions, elapsed time, speed/ETA, current question, tool turns and running measured reward. The bar advances only after an episode completes.


In [7]:
from kg_rlvr.training.rollouts import evaluate_policy
BASE_METRICS, BASE_TRACES = None, []
if RUN_MODEL:
    with model.disable_adapter():
        BASE_METRICS, BASE_TRACES = evaluate_variant(model, tokenizer, store, test_ids, LIMITS,
            RUN_ROOT, "vanilla", show_progress=SHOW_PROGRESS)
    display(pd.Series(BASE_METRICS, name="measured vanilla test performance").to_frame())
    MEMORY.append(memory_snapshot("after_baseline"))
    save_json(VARIANT_DIRS["vanilla"] / "memory_snapshots.json", [MEMORY[-1]])
    if not any(event["name"] == "get_neighbors" and event["observation"].get("ok") for trace in BASE_TRACES for event in trace["tool_events"]):
        print("Vanilla executed no valid neighbor tools. No working baseline tool use is claimed; supervised demonstrations will provide valid action targets.")
else:
    print("Vanilla test evaluation not run; no baseline accuracy is reported.")


Vanilla test evaluation not run; no baseline accuracy is reported.


### Inspect vanilla model outputs and tool calls

Each card shows the input messages, expandable full Qwen input including tool schemas, exact assistant output, parsed tool arguments, and the actual tool response for every turn. The submission acknowledgment is shown even when it was not appended to the model context. Change `INSPECT_EXAMPLES` and `INSPECT_OFFSET` and rerun this cell to browse the stored episodes; this does not call the model again. Examples follow stored order, including failures.


In [8]:
from IPython.display import HTML
from kg_rlvr.training.inspection import render_traces_html

display(HTML(render_traces_html(BASE_TRACES, label="Vanilla baseline · test",
    count=INSPECT_EXAMPLES, offset=INSPECT_OFFSET, tokenizer=tokenizer)))


<a id="sft"></a>
## 6b. Supervised tool-use warm start

Train for one pass on 200 reserved demonstrations, with 20%/40%/40% one/two/three-hop questions. Every reference trajectory executes `get_relations`, `get_neighbors` and a correct `submit_answer` through the real environment, using the same limits as model rollouts. These are **reference demonstrations, not model-generated results**.

Only assistant tool-call tokens and EOS receive SFT loss. The initial prompt, actual tool observations, injected assistant prefixes and padding are masked with `-100`. The exact Qwen runtime codec is reused, with no fabricated observations or silent truncation. No chain-of-thought text is added.

Training uses the existing NF4 base and LoRA adapter with microbatch 1, accumulation 8 and learning rate 2e-5. One pass over 200 examples produces 25 SFT optimizer updates. Actual gradients, weight changes and resumable checkpoint files must pass verification before continuing. The SFT adapter is exported separately, then its optimizer is released before GRPO.

The warm-start manifest fingerprints the SFT adapter; incompatible resumed runs fail explicitly. Both GRPO phases require a verified SFT result and exclude the SFT query instances. An independent GRPO-only branch follows the SFT + GRPO export.


In [9]:
from kg_rlvr.training.sft import encode_demonstrations, build_sft_trainer
from kg_rlvr.training.grpo import lora_snapshot, verify_optimizer_update, export_adapter, training_manifest
SFT_RESULT, sft_trainer = None, None
SFT_WARM_START = {"enabled": False, "reserved_sft_ids": sft_ids, "stage_partition_sha256": STAGE_PARTITION_SHA256}
SFT_CONFIG = {"epochs": SFT_EPOCHS, "examples": SFT_EXAMPLES, "learning_rate": SFT_LEARNING_RATE,
    "accumulation": SFT_GRADIENT_ACCUMULATION_STEPS, "loss": "assistant_only_causal_lm"}
if RUN_MODEL:
    sft_dir = CHECKPOINT_ROOT / "sft"
    if sft_dir.exists() and any(sft_dir.iterdir()):
        raise FileExistsError("SFT output already exists; choose a fresh RUN_NAME to preserve measured evidence")
    sft_rows = encode_demonstrations(store, SFT_DEMONSTRATIONS, tokenizer, LIMITS, show_progress=SHOW_PROGRESS)
    sft_manifest = {"training_kind": "SFT", "model_name": MODEL_NAME, "revision": MODEL_REVISION,
        "task_ids": sft_ids, "configuration": SFT_CONFIG, "stage_partition_sha256": STAGE_PARTITION_SHA256,
        "source_sha256": INPUT_SHA256, "trainable_parameter_dtype": "torch.float32",
        "implementation_sha256": training_manifest(MODEL_NAME, MODEL_REVISION, LIMITS, sft_ids, 8, 8,
            SFT_LEARNING_RATE, SEED)["implementation_sha256"]}
    sft_dir.mkdir(parents=True, exist_ok=True)
    (sft_dir / "run_manifest.json").write_text(json.dumps(sft_manifest, indent=2) + "\n")
    sft_trainer = build_sft_trainer(model, tokenizer, sft_rows, sft_dir, epochs=SFT_EPOCHS,
        accumulation=SFT_GRADIENT_ACCUMULATION_STEPS, learning_rate=SFT_LEARNING_RATE, seed=SEED,
        show_progress=SHOW_PROGRESS, diagnostics_dir=SFT_RUN_DIR / "training/sft")
    before_sft = lora_snapshot(model)
    torch.cuda.reset_peak_memory_stats()
    try:
        sft_train_result = sft_trainer.train()
        required_sft_steps = math.ceil(math.ceil(len(sft_rows) / SFT_GRADIENT_ACCUMULATION_STEPS) * SFT_EPOCHS)
        SFT_RESULT = verify_optimizer_update(model, before_sft, sft_trainer, sft_dir,
            required_steps=required_sft_steps, training_kind="SFT")
        SFT_RESULT.update(train_loss=float(sft_train_result.training_loss), examples=len(sft_rows), epochs=SFT_EPOCHS)
        model = sft_trainer.accelerator.unwrap_model(sft_trainer.model)
        export_adapter(model, tokenizer, SFT_RUN_DIR / "adapter", sft_manifest, extra={"verified_result": SFT_RESULT})
        SFT_WARM_START = {"enabled": True, "reserved_sft_ids": sft_ids,
            "stage_partition_sha256": STAGE_PARTITION_SHA256, "configuration": SFT_CONFIG,
            "adapter_sha256": {name: digest for name, digest in fingerprint_inputs(SFT_RUN_DIR / "adapter").items()
                               if name in {"adapter_config.json", "adapter_model.safetensors"}}}
        (SFT_RUN_DIR / "training_result.json").write_text(json.dumps(SFT_RESULT, indent=2) + "\n")
    except Exception as error:
        save_json(SFT_RUN_DIR / "status.json", {"variant": "sft", "status": "training_failed", "error": repr(error)})
        (SFT_RUN_DIR / "failure.json").write_text(json.dumps({"error": repr(error), "configuration": SFT_CONFIG}, indent=2) + "\n")
        raise  # never start GRPO after failed supervised training
    finally:
        collect_training_logs(SFT_RUN_DIR, [sft_dir])
        del before_sft
    for name in ["trainer_metrics.jsonl", "step_metrics.jsonl", "precision_checks.jsonl", "run_manifest.json"]:
        if (sft_dir / name).is_file():
            shutil.copy2(sft_dir / name, SFT_RUN_DIR / name)
    sft_logs = pd.DataFrame([row for row in sft_trainer.state.log_history if "loss" in row])
    display(sft_logs)
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(sft_logs.step, sft_logs.loss, marker=".")
    ax.set(title="SFT assistant-token loss", xlabel="SFT optimizer step", ylabel="Cross-entropy loss")
    (SFT_RUN_DIR / "figures").mkdir(exist_ok=True)
    fig.savefig(SFT_RUN_DIR / "figures/sft_loss.png", bbox_inches="tight"); plt.show(); plt.close(fig)
    sft_trainer.optimizer = None; sft_trainer.lr_scheduler = None
    sft_trainer.accelerator.free_memory()
    sft_trainer = None
    model.zero_grad(set_to_none=True)
    del sft_rows
    gc.collect(); torch.cuda.empty_cache()
    MEMORY.append(memory_snapshot("after_verified_sft"))
    display(pd.Series(SFT_RESULT, name="verified supervised warm start").to_frame())
else:
    print("SFT training not executed in CPU preflight. GRPO requires a verified SFT warm start.")


SFT training not executed in CPU preflight. GRPO requires a verified SFT warm start.


## 6c. SFT test evaluation and tool inspection

Evaluate the verified SFT policy on the exact same ordered test IDs as vanilla, before any GRPO update. Cards show the original prompt, actual model tool calls and tool observations. SFT and GRPO use disjoint train-only instances; test examples never enter either optimization dataset.


In [10]:
SFT_METRICS, SFT_TRACES = None, []
if RUN_MODEL and SFT_RESULT is not None:
    SFT_METRICS, SFT_TRACES = evaluate_variant(model, tokenizer, store, test_ids, LIMITS,
        RUN_ROOT, "sft", show_progress=SHOW_PROGRESS, training_result=SFT_RESULT)
    display(pd.Series(SFT_METRICS, name="measured SFT test performance").to_frame())
    display(HTML(render_traces_html(SFT_TRACES, label="SFT · test", count=INSPECT_EXAMPLES,
        offset=INSPECT_OFFSET, tokenizer=tokenizer)))
    MEMORY.append(memory_snapshot("after_sft_only_evaluation"))
    save_json(SFT_RUN_DIR / "memory_snapshots.json", [MEMORY[-1]])
else:
    print("SFT-only evaluation not run; no SFT performance claim is available.")


SFT-only evaluation not run; no SFT performance claim is available.


<a id="grpo"></a>
## 7. GRPO configuration and rollout integration

The native TRL 1.8.0 environment loop was inspected. It does not consult per-episode completion after submission, so this notebook uses its supported **custom `rollout_func`** instead. The deviation is orchestration only: `GRPOTrainer` still computes group-normalized advantages, the GRPO clipped objective, gradients, and optimizer steps.

- TRL 1.8.0's actual RepeatSampler already supplies `G` consecutive copies of each prompt to this hook. We assert identical prompts and fresh state within every group and **do not repeat them again**. The version is pinned because this experimental API can change.
- Raw sampled action IDs are preserved, including EOS. Tool suffixes are rendered with the real Qwen template and concatenated, without re-tokenizing or canonicalizing generated actions. `env_mask=1` marks model tokens; `0` marks observations and injected assistant prefixes. TRL consumes this mask as its tool loss mask. Observations remain attention context but are excluded from the policy objective.
- The custom hook returns `logprobs=None` as supported by the inspected release; TRL computes on-policy log-probabilities. Sampling uses temperature 0.8, no top-k/p warping, and fresh GenerationConfig defaults, matching the trainer's temperature-scaled policy probabilities.
- One GPU × microbatch 1 × accumulation 8 = eight completion trajectories per optimizer update. With `G=8`, that is **one unique question/group per update**. Accumulation 16 gives two groups per update. Generation batch size equals accumulation; `num_iterations=1` keeps updates on-policy.
- AdamW updates only the small FP32 LoRA parameters, with learning rate 5e-6, no weight decay, constant scheduling and gradient norm cap 1. There is **no KL reference model** (`beta=0`) and no KL metric. This avoids an unnecessary base-model copy. Zero-reward or all-success groups have zero reward variance and zero policy signal; both are logged.

No native `tools` or `environment_factory` argument is passed alongside the custom hook, which would execute calls a second time. Unsupported legacy config arguments are rejected. Periodic evaluation is a separate greedy callback, so GRPO generation-group batching is not confused with evaluation batching.
**Live progress:** `SHOW_PROGRESS=True` keeps the Trainer's optimizer-step progress enabled and shows a separate trajectory counter while GRPO samples each comparison batch. A batch of eight trajectories is shown as eight trajectories; optimizer steps are counted by the Trainer. Tool-turn/token status updates during sampling. Model downloads/weight loading use the native Hugging Face bars. Progress can be disabled with the same configuration switch. The shared `tqdm` dependency comes from TRL/datasets and automatically selects notebook or terminal display ([tqdm documentation](https://tqdm.github.io/docs/shortcuts/#tqdmauto)).

**Evaluation rewards during training:** by default, both smoke and full phases evaluate the same first `PERIODIC_EVAL_SIZE=6` fixed test questions after every optimizer step (`EVALUATE_EVERY_STEPS=1`). Increase the interval to evaluate less frequently, or use 0 to disable it. Evaluation adds real inference time and uses the same greedy policy/tool limits as the baseline; it never supplies rewards to the policy or optimizer. One binary answer correctness is computed for each completed question, rather than for each tool call. The progress bar shows the latest reward and running mean; each evaluation prints mean/total reward and per-hop means. Training rollout rewards and test rewards are stored and plotted separately.


The optimizer receives `answer_correct + 0.2 * gold_entity_retrieved`. The retrieval bonus is awarded once, only for IDs in successful executed neighbor responses independently checked against the graph. It is never awarded for model prose, relation lists, failed queries, repeated retrievals, or acknowledgments. Gold answers and scores are private to post-episode verification. Binary answer reward/accuracy remains the evaluation measure; logs also include retrieval success, retrieval bonus and combined training reward.

With G=8, accumulation=8 and microbatch 1, each optimizer step samples eight trajectories for one question. The ordered weighted schedule chooses that question's hop category; TRL supplies the eight repetitions. Every five groups have 1/2/2 questions by hop (30 steps: 6/12/12). A partial block uses integer rounding (8 smoke steps: 2/3/3). Questions within a hop are shuffled and cycled if the requested exposure exceeds the selected pool. The planned schedule and actual trajectory hop counts are saved for audit and resume.


In [11]:
from kg_rlvr.training.grpo import (
    build_trainer, lora_snapshot, verify_optimizer_update, training_manifest,
    check_resume, export_adapter, reload_adapter_in_place, assert_lora_fp32,
)
EXPERIMENT_CONFIG = {"model_name": MODEL_NAME, "revision": MODEL_REVISION,
    "train_hops": TRAIN_HOPS, "eval_hops": EVAL_HOPS, "num_generations": NUM_GENERATIONS,
    "sft": {"enabled": True, "examples": SFT_EXAMPLES, "epochs": SFT_EPOCHS,
        "learning_rate": SFT_LEARNING_RATE, "accumulation": SFT_GRADIENT_ACCUMULATION_STEPS},
    "stage_partition_sha256": STAGE_PARTITION_SHA256, "supervised_warm_start": SFT_WARM_START,
    "evaluation_split": "test", "evaluation_task_ids": test_ids,
    "comparison_variants": ["vanilla", "sft", "sft_grpo", "grpo_only"],
    "train_hop_exposure": TRAIN_HOP_EXPOSURE, "single_answer_only": SINGLE_ANSWER_ONLY, "reward_spec": REWARD_SPEC,
    "microbatch": 1, "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "unique_questions_per_update": GRADIENT_ACCUMULATION_STEPS // NUM_GENERATIONS,
    "learning_rate": LEARNING_RATE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "lora_dropout": 0.0, "limits": asdict(LIMITS), "smoke_steps": SMOKE_TEST_STEPS,
    "full_steps": INITIAL_TRAINING_STEPS, "save_steps": SAVE_STEPS,
    "evaluate_every_steps": EVALUATE_EVERY_STEPS, "periodic_eval_size": PERIODIC_EVAL_SIZE, "seed": SEED, "show_progress": SHOW_PROGRESS}
display(pd.Series(EXPERIMENT_CONFIG, name="requested configuration").to_frame())
(RUN_ROOT / "experiment_config.json").write_text(json.dumps(EXPERIMENT_CONFIG, indent=2) + "\n")
smoke_trainer, full_trainer, final_trainer = None, None, None
SMOKE_RESULT, FULL_RESULT = None, None
from kg_rlvr.training.experiments import run_rl_phase, release_trainer
from kg_rlvr.training.artifacts import persist_rl_diagnostics
RL_KWARGS = {"generations": NUM_GENERATIONS, "accumulation": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE, "seed": SEED, "test_ids": test_ids[:PERIODIC_EVAL_SIZE],
    "eval_every": EVALUATE_EVERY_STEPS, "show_progress": SHOW_PROGRESS,
    "hop_weights": TRAIN_HOP_EXPOSURE, "excluded_sft_task_ids": sft_ids}
def rl_manifest(variant, warm_start):
    manifest = training_manifest(MODEL_NAME, MODEL_REVISION, LIMITS, train_ids, NUM_GENERATIONS,
        GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, SEED, LORA_R, LORA_ALPHA,
        hop_weights=TRAIN_HOP_EXPOSURE, single_answer_only=SINGLE_ANSWER_ONLY, supervised_warm_start=warm_start)
    manifest.update(variant=variant, source_sha256=INPUT_SHA256, initial_lora_sha256=INITIAL_LORA_SHA256,
        evaluation_split="test", evaluation_task_ids=test_ids)
    return manifest


,requested configuration
model_name,Qwen/Qwen3-1.7B
revision,70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
train_hops,"[1, 2, 3]"
eval_hops,"[1, 2, 3, 4]"
num_generations,8
sft,"{'enabled': True, 'examples': 200, 'epochs': 1..."
stage_partition_sha256,21122cbedd6a82f703768c027ef1698370fda7428dadcd...
supervised_warm_start,"{'enabled': False, 'reserved_sft_ids': ['q_000..."
evaluation_split,test
evaluation_task_ids,"[q_003492, q_002025, q_002129, q_002709, q_003..."


<a id="smoke"></a>
## 8. Genuine GRPO smoke test

This is not an inference-only test. The real model samples multiple independent episodes, calls graph tools, receives verified correctness-plus-retrieval rewards, and runs `trainer.train()`. The gate requires finite logged losses, a positive finite gradient norm, changed trainable LoRA tensors, a saved adapter plus optimizer/scheduler/trainer/RNG checkpoint, and measured peak GPU memory.

The default smoke budget is eight optimizer-step attempts, covering eight groups and 64 sampled episodes with `NUM_GENERATIONS=8` and accumulation 8. This increases sampling over the previous two-step default, but cannot guarantee mixed rewards or learning. The shared public prompt asks the model to finish all relationships and check the requested answer type. Baseline and training use the same prompt. No per-question answer type or path is supplied from verifier metadata.

The smoke pool is a seeded, balanced sample across the configured **training** hop categories (default `[1, 2, 3]`). This includes easier one-hop tasks alongside multi-hop tasks. Full training uses the full selected mixed training pool; `MAX_TRAIN_EXAMPLES` caps that pool. Each GRPO group still samples one question independently; different hop categories are never compared within the same group. No scripted actions or reference programs guide model rollouts.

If no LoRA tensor changes, the smoke fails even if inference and optimizer plumbing ran. Inspect zero-variance groups, raw actions, malformed calls, submission rate, and truncation. Homogeneous combined training rewards legitimately provide no gradient, including groups at 0.2 or 1.2. Inspect the separate answer and retrieval components. Increase real smoke groups or adjust explicit limits/sampling after diagnosis; no manufactured successes are inserted. Full training must not proceed after failure.
**QLoRA precision check:** TRL 1.8.0 casts trainable quantized-model parameters to BF16 in its constructor even when FP16 is selected. `build_trainer` therefore restores FP32 LoRA storage **after construction and before optimizer creation**, preserving the adapter values exactly. The frozen NF4 base is unchanged. FP16/BF16 here describes computation; LoRA optimizer parameters remain FP32. Checks run again after Trainer/Accelerate preparation or checkpoint loading and before each optimizer-step cycle. The actual flags and adapter dtype are displayed below and saved in `precision_checks.jsonl`.

If an earlier smoke failed with `_amp_foreach_non_finite_check_and_unscale_cuda ... BFloat16`, restart the Python kernel, use the updated repository code and a fresh `RUN_NAME`, then rerun initialization and smoke. Existing gradients/scaler state from the failed run must not be reused. Keep the T4 FP16 setting; changing the clipping threshold or hiding the experimental-API warning does not repair parameter precision.

On failure, raw phase logs, a failure/status record, episodic returns and HTML trajectories are preserved under the variant directory before the original error is re-raised. Inspect these files without requiring a live trainer. The final ZIP cell can still be run to download partial diagnostics. All-zero groups mean exploration produced no correct answers; mixed groups with unchanged weights require investigation of gradients/optimizer state instead.

After updating the prompt, restart the kernel with the updated source and choose a fresh `RUN_NAME`, for example `kg_grpo_multihop_seed41`. Recompute the vanilla baseline before training; do not compare a new trained prompt against an old baseline prompt. The GitHub setup reuses an existing checkout without pulling, so update that checkout explicitly first.


In [12]:
if RUN_MODEL and RUN_SMOKE_TEST:
    if SFT_RESULT is None:
        raise RuntimeError("SFT + GRPO requires the SFT stage to finish and verify its optimizer update first")
    smoke_manifest = rl_manifest("sft_grpo", SFT_WARM_START)
    smoke_trainer, smoke_phase_result = run_rl_phase(model, tokenizer, store, train_ids, LIMITS,
        variant_dir=VARIANT_DIRS["sft_grpo"], checkpoint_root=SFT_GRPO_CHECKPOINT_ROOT,
        phase="smoke", steps=SMOKE_TEST_STEPS, manifest=smoke_manifest, save_steps=SMOKE_TEST_STEPS,
        **RL_KWARGS)
    SMOKE_RESULT = smoke_phase_result["verified_update"]
    final_trainer = smoke_trainer
    MEMORY.append(memory_snapshot("after_sft_grpo_smoke"))
    display(pd.Series(SMOKE_RESULT, name="verified SFT + GRPO smoke").to_frame())
else:
    print("SFT + GRPO smoke not run; no RL update is claimed.")


SFT + GRPO smoke not run; no RL update is claimed.


<a id="full"></a>
## 9. Full training and checkpoint resumption

Enable `RUN_FULL_TRAINING=True` to use the configured real training pool for the initial 30-step run. The smoke adapter is the warm start, while this phase intentionally creates a fresh optimizer/scheduler. Its baseline comparison therefore includes smoke updates. When resuming, the full checkpoint overrides that warm start with its saved adapter, optimizer, scheduler, RNG and trainer state.

`INITIAL_TRAINING_STEPS` is the Trainer's **absolute full-phase optimizer-step target**, not additional steps after a resume. `RESUME_FROM_CHECKPOINT` must point to a complete `checkpoint-*` directory accompanied by the original `run_manifest.json`. Adapter-only exports are inference artifacts and do not provide exact resumption. The manifest checks source fingerprints, selected training IDs, model/revision, versions, LoRA and rollout settings. Extending the target is supported by the constant scheduler; GPU kernels/session differences can still prevent bitwise-identical continuation.

Checkpoints are saved under `/kaggle/working/checkpoints/<RUN_NAME>/full/`, retaining two. A non-resume run refuses to mix logs with an earlier run. Periodic test monitoring runs after every optimizer step by default; its interval is configurable.
Set full training before Run All when ready; the smoke gate still runs first. After a completed smoke-only Run All, restart with a fresh `RUN_NAME` and full training enabled. Export/reload has already released the old trainers and made that adapter inference-only. To continue within the initial session, stop after section 8, set `RUN_FULL_TRAINING=True`, then execute sections 9 onward before export.


In [13]:
if RUN_MODEL and RUN_FULL_TRAINING:
    if SFT_RESULT is None:
        raise RuntimeError("SFT + GRPO requires the SFT stage to finish and verify its optimizer update first")
    if SMOKE_RESULT is None or smoke_trainer is None:
        raise RuntimeError("Full SFT + GRPO requires this run's verified smoke phase")
    model = release_trainer(smoke_trainer)
    smoke_trainer = final_trainer = None
    gc.collect(); torch.cuda.empty_cache()
    full_manifest = rl_manifest("sft_grpo", SFT_WARM_START)
    full_trainer, FULL_RESULT = run_rl_phase(model, tokenizer, store, train_ids, LIMITS,
        variant_dir=VARIANT_DIRS["sft_grpo"], checkpoint_root=SFT_GRPO_CHECKPOINT_ROOT,
        phase="full", steps=INITIAL_TRAINING_STEPS, manifest=full_manifest, save_steps=SAVE_STEPS,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT, **RL_KWARGS)
    final_trainer = full_trainer
    MEMORY.append(memory_snapshot("after_sft_grpo_full"))
    display(pd.Series(FULL_RESULT, name="measured SFT + GRPO full phase").to_frame())
else:
    print("Full SFT + GRPO not run. The measured smoke policy is retained when available.")


Full SFT + GRPO not run. The measured smoke policy is retained when available.


### Inspect sampled training episodes

These are the actual sampled trajectories used by GRPO, rather than greedy evaluation outputs. Full-phase traces are shown when available, otherwise smoke traces. Rewards are post-rollout diagnostics; tool responses never reveal correctness. This cell reads saved JSONL and works without a live trainer. Training failures can also be inspected by running this cell manually after the exception.


In [14]:
from IPython.display import HTML
from kg_rlvr.training.inspection import render_traces_html
training_traces = read_rows(VARIANT_DIRS["sft_grpo"] / "training_trajectories.jsonl")
display(HTML(render_traces_html(training_traces, label="SFT + GRPO · sampled training episodes",
    count=INSPECT_EXAMPLES, offset=INSPECT_OFFSET, tokenizer=tokenizer)))


<a id="metrics"></a>
## 10. Actual rollout, optimizer, and resource metrics

Reward and success plots are read from real rollout JSONL records. Zero-reward groups and all-homogeneous groups are separate: both all-zero and all-one groups have zero variance. Call rates count attempted calls; malformed calls are invalid attempts. Episodes with no calls have a valid-call rate of zero by definition. Completion here means a submitted entity, with other termination reasons tabulated separately. Gold reward is answer accuracy under the single-answer task.

KL divergence is unavailable because `beta=0`. No exploration cost or GPU measurement is fabricated for a skipped phase. Wall time includes actual rollout/forward/backward work. Baseline/trained trajectories and tool observations are private diagnostics, not model prompt columns.
`mean_episode_reward` is the combined training reward (0–1.2); `accuracy`/`mean_reward` remain binary answer correctness. Zero-variance metrics use the actual combined rewards supplied to GRPO, with a separate `answer_zero_variance_group_fraction` for binary correctness. Retrieval success is reported separately.


In [15]:
from IPython.display import Image
sft_grpo_dir = VARIANT_DIRS["sft_grpo"]
rollout_metrics = pd.DataFrame(read_rows(sft_grpo_dir / "rollout_metrics.jsonl"))
trainer_metrics = pd.DataFrame(read_rows(sft_grpo_dir / "trainer_metrics.jsonl"))
step_metrics = pd.DataFrame(read_rows(sft_grpo_dir / "step_metrics.jsonl"))
if not rollout_metrics.empty:
    display(rollout_metrics)
    for name in ["rollout_metrics.png", "optimizer_metrics.png"]:
        path = sft_grpo_dir / "figures" / name
        if path.is_file():
            display(Image(filename=str(path)))
else:
    print("No SFT + GRPO rollout data exists; no metrics are fabricated.")
display(pd.DataFrame(MEMORY))


No SFT + GRPO rollout data exists; no metrics are fabricated.


,stage,system_available_gib,cuda_available,allocated_gib,reserved_gib,peak_allocated_gib,peak_reserved_gib
0,before_model,0.725754,False,None,None,None,None


### Test rewards at each training checkpoint

The summary table and curve contain measured test rewards at each evaluated optimizer step. The question table includes every evaluated question's terminal reward, hop count, submitted ID and termination reason. `evaluation_metrics` and `evaluation_rewards` remain available for filtering by phase/step. A missing reward is never filled with zero. With the binary verifier, mean reward equals answer accuracy.

Evaluation `reward` and `mean_reward` remain binary answer correctness. Each question also records `gold_entity_retrieved`, `retrieval_reward`, and `training_reward`; summaries expose retrieval success and mean combined reward. These diagnostics never update the optimizer.

These test evaluations are monitoring only: they do not supply gradients, select checkpoints, trigger early stopping or change the fixed training schedule. They reuse a subset of the common test IDs.


In [16]:
evaluation_metrics = pd.DataFrame(read_rows(sft_grpo_dir / "evaluation_metrics.jsonl"))
evaluation_rewards = pd.DataFrame(read_rows(sft_grpo_dir / "evaluation_rewards.jsonl"))
if not evaluation_metrics.empty:
    display(evaluation_metrics)
    display(evaluation_rewards)
    path = sft_grpo_dir / "figures/evaluation_rewards.png"
    if path.is_file():
        display(Image(filename=str(path)))
else:
    print("No periodic SFT + GRPO test evaluations recorded.")


No periodic SFT + GRPO test evaluations recorded.


<a id="evaluation"></a>
## 11. Save SFT + GRPO test performance

Run the SFT + GRPO policy on the same ordered test questions already evaluated by vanilla and SFT. Metrics, raw trajectories, rewards and HTML cards are written to `variants/sft_grpo/` immediately. The independent GRPO-only stage comes after this policy has been exported.


In [17]:
SFT_GRPO_METRICS, SFT_GRPO_TRACES = None, []
if RUN_MODEL and SMOKE_RESULT is not None:
    if SFT_RESULT is None:
        raise RuntimeError("SFT + GRPO evaluation requires a verified SFT warm start")
    model = final_trainer.accelerator.unwrap_model(final_trainer.model)
    manifest = full_manifest if FULL_RESULT is not None else smoke_manifest
    export_adapter(model, tokenizer, ADAPTER_DIR, manifest,
        extra={"experiment_config": EXPERIMENT_CONFIG, "smoke_verified": SMOKE_RESULT, "full_training_result": FULL_RESULT})
    SFT_GRPO_METRICS, SFT_GRPO_TRACES = evaluate_variant(model, tokenizer, store, test_ids, LIMITS,
        RUN_ROOT, "sft_grpo", show_progress=SHOW_PROGRESS,
        training_result={"smoke": SMOKE_RESULT, "full": FULL_RESULT})
    display(pd.Series(SFT_GRPO_METRICS, name="measured SFT + GRPO test").to_frame())
    MEMORY.append(memory_snapshot("after_sft_grpo_evaluation"))
else:
    print("SFT + GRPO test evaluation not run; no accuracy is claimed.")


SFT + GRPO test evaluation not run; no accuracy is claimed.


### Inspect SFT + GRPO outputs and tool calls

These cards use the same ordered test IDs as vanilla and SFT. Matching inspection offsets show matching questions across all four policies. The cards display stored inference; no additional generation occurs here.


In [18]:
from IPython.display import HTML
from kg_rlvr.training.inspection import render_traces_html

display(HTML(render_traces_html(SFT_GRPO_TRACES, label="SFT + GRPO · test",
    count=INSPECT_EXAMPLES, offset=INSPECT_OFFSET, tokenizer=tokenizer)))


<a id="export"></a>
## 12. Adapter export, reload, and download

The adapter-only export goes to `RUN_ROOT / "variants/sft_grpo/adapter"`, with tokenizer files, immutable model revision, training configuration, dependency versions and references to actual logs/evaluations. There is no unnecessary full base-model copy. Full trainer checkpoints remain separate under `checkpoints/` and include optimizer/scheduler/RNG state.

Before the reload test, optimizer/trainer references are released. The adapter is unloaded and reloaded from disk onto the **same quantized base weights**. A fresh graph-tool evaluation of three fixed test questions checks the reloaded policy. The ZIP includes adapter and run diagnostics, excluding large trainer checkpoints and Hub caches. Save checkpoints separately if you intend to resume.

For later inference, load the same model/revision using `AutoModelForCausalLM.from_pretrained(..., quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16), device_map={"":0})`, load tokenizer files from the adapter directory, then call `PeftModel.from_pretrained(base, adapter_directory, is_trainable=False)`. The exported manifest records the exact revision and limits. To reproduce graph interaction, use this repository's `evaluate_policy`/rollout engine with that policy and the saved graph.

In [19]:
RELOAD_METRICS, RELOAD_TRACES = None, []
if RUN_MODEL and SMOKE_RESULT is not None:
    manifest = full_manifest if FULL_RESULT is not None else smoke_manifest
    export_adapter(model, tokenizer, ADAPTER_DIR, manifest, extra={"experiment_config": EXPERIMENT_CONFIG,
        "final_phase": "full" if FULL_RESULT is not None else "smoke", "smoke_verified": SMOKE_RESULT,
        "full_training_result": FULL_RESULT, "sft_result": SFT_RESULT, "vanilla_test_metrics": BASE_METRICS, "sft_test_metrics": SFT_METRICS,
        "stage_partition_sha256": STAGE_PARTITION_SHA256, "hardware": hardware,
        "sft_grpo_test_metrics": SFT_GRPO_METRICS, "evaluation_split": "test", "evaluation_task_ids": test_ids,
        "training_logs_directory": str(CHECKPOINT_ROOT), "evaluation_directory": str(EVALUATION_DIR)})
    for trainer in [smoke_trainer, full_trainer]:
        if trainer is not None:
            trainer.optimizer = None; trainer.lr_scheduler = None
            trainer.accelerator.free_memory()
    smoke_trainer = full_trainer = final_trainer = None
    if "trainer" in globals():
        del trainer
    gc.collect(); torch.cuda.empty_cache()
    model = reload_adapter_in_place(model, ADAPTER_DIR)
    RELOAD_METRICS, RELOAD_TRACES = evaluate_policy(model, tokenizer, store, test_ids[:RELOAD_EVAL_SIZE], LIMITS, EVALUATION_DIR, "reloaded_adapter", show_progress=SHOW_PROGRESS)
    save_json(VARIANT_DIRS["sft_grpo"] / "memory_snapshots.json", MEMORY)
    display(pd.Series(RELOAD_METRICS, name="measured adapter reload evaluation").to_frame())
    MEMORY.append(memory_snapshot("after_adapter_reload_evaluation"))
    print("Adapter-only export:", ADAPTER_DIR, "Full checkpoints:", CHECKPOINT_ROOT)
else:
    print("No GRPO adapter exported or reloaded: GPU smoke training has not been verified. Any verified SFT adapter is under RUN_ROOT/sft/adapter.")

No GRPO adapter exported or reloaded: GPU smoke training has not been verified. Any verified SFT adapter is under RUN_ROOT/sft/adapter.


<a id="grpo-only"></a>
## 13. Independent GRPO-only variant

Restore the exact saved **untrained vanilla adapter** onto the same frozen quantized base, with trainable FP32 LoRA parameters. The value fingerprint must match its pre-SFT initialization. SFT and previous RL weights and optimizer states are not reused. This avoids holding two large models in GPU memory.

GRPO-only uses the same 733-question GRPO pool, excludes all 200 reserved SFT instances, and uses the same hop schedule, seed, G, accumulation, reward, limits, smoke/full budget and fixed test IDs as SFT + GRPO. `RUN_GRPO_ONLY=True` enables this fourth variant. Resume paths are separate for each RL variant.


In [20]:
GRPO_ONLY_SMOKE, GRPO_ONLY_FULL, GRPO_ONLY_METRICS, GRPO_ONLY_TRACES = None, None, None, []
g_only_smoke_trainer = g_only_full_trainer = g_only_final_trainer = None
if RUN_MODEL and RUN_GRPO_ONLY:
    for previous_trainer in [smoke_trainer, full_trainer, final_trainer]:
        if previous_trainer is not None:
            model = release_trainer(previous_trainer)
    previous_trainer = None
    smoke_trainer = full_trainer = final_trainer = None
    gc.collect(); torch.cuda.empty_cache()
    model = reload_adapter_in_place(model, INITIAL_ADAPTER_DIR, is_trainable=True)
    assert_lora_fp32(model)
    if lora_fingerprint(model) != INITIAL_LORA_SHA256:
        raise RuntimeError("GRPO-only did not restore the exact pre-SFT adapter initialization")
    GRPO_ONLY_WARM_START = {"enabled": False, "reserved_sft_ids": sft_ids,
        "stage_partition_sha256": STAGE_PARTITION_SHA256, "initial_lora_sha256": INITIAL_LORA_SHA256}
    save_json(VARIANT_DIRS["grpo_only"] / "initialization.json", GRPO_ONLY_WARM_START)
    print("GRPO-only restored the exact vanilla initialization; no SFT adapter weights retained.")
else:
    print("GRPO-only model initialization skipped.")


GRPO-only model initialization skipped.


### GRPO-only smoke training

Use the same verified update gate as SFT + GRPO. Phase logs, episodic returns, charts and HTML trajectories are saved inside `variants/grpo_only/` when this phase finishes, including on failure.


In [21]:
if RUN_MODEL and RUN_GRPO_ONLY and RUN_SMOKE_TEST:
    g_only_smoke_manifest = rl_manifest("grpo_only", GRPO_ONLY_WARM_START)
    g_only_smoke_trainer, g_only_smoke_result = run_rl_phase(model, tokenizer, store, train_ids, LIMITS,
        variant_dir=VARIANT_DIRS["grpo_only"], checkpoint_root=GRPO_ONLY_CHECKPOINT_ROOT,
        phase="smoke", steps=SMOKE_TEST_STEPS, manifest=g_only_smoke_manifest, save_steps=SMOKE_TEST_STEPS,
        **RL_KWARGS)
    GRPO_ONLY_SMOKE = g_only_smoke_result["verified_update"]
    g_only_final_trainer = g_only_smoke_trainer
    display(pd.Series(GRPO_ONLY_SMOKE, name="verified GRPO-only smoke").to_frame())
else:
    print("GRPO-only smoke not run.")


GRPO-only smoke not run.


In [22]:
if RUN_MODEL and RUN_GRPO_ONLY and RUN_FULL_TRAINING:
    if GRPO_ONLY_SMOKE is None or g_only_smoke_trainer is None:
        raise RuntimeError("Full GRPO-only training requires its own verified smoke update")
    model = release_trainer(g_only_smoke_trainer)
    g_only_smoke_trainer = g_only_final_trainer = None
    gc.collect(); torch.cuda.empty_cache()
    g_only_full_manifest = rl_manifest("grpo_only", GRPO_ONLY_WARM_START)
    g_only_full_trainer, GRPO_ONLY_FULL = run_rl_phase(model, tokenizer, store, train_ids, LIMITS,
        variant_dir=VARIANT_DIRS["grpo_only"], checkpoint_root=GRPO_ONLY_CHECKPOINT_ROOT,
        phase="full", steps=INITIAL_TRAINING_STEPS, manifest=g_only_full_manifest, save_steps=SAVE_STEPS,
        resume_from_checkpoint=GRPO_ONLY_RESUME_FROM_CHECKPOINT, **RL_KWARGS)
    g_only_final_trainer = g_only_full_trainer
    display(pd.Series(GRPO_ONLY_FULL, name="measured GRPO-only full phase").to_frame())
else:
    print("Full GRPO-only training not run; the measured smoke policy is retained when available.")


Full GRPO-only training not run; the measured smoke policy is retained when available.


### Save GRPO-only performance and adapter

The same test IDs are evaluated before any model is released. Metrics, per-question rewards, raw trajectories, HTML cards and the independent adapter are saved in `variants/grpo_only/` now. A later notebook failure does not discard completed variants.


In [23]:
if RUN_MODEL and RUN_GRPO_ONLY and GRPO_ONLY_SMOKE is not None:
    model = g_only_final_trainer.accelerator.unwrap_model(g_only_final_trainer.model)
    manifest = g_only_full_manifest if GRPO_ONLY_FULL is not None else g_only_smoke_manifest
    export_adapter(model, tokenizer, VARIANT_DIRS["grpo_only"] / "adapter", manifest,
        extra={"experiment_config": EXPERIMENT_CONFIG, "smoke_verified": GRPO_ONLY_SMOKE, "full_training_result": GRPO_ONLY_FULL})
    GRPO_ONLY_METRICS, GRPO_ONLY_TRACES = evaluate_variant(model, tokenizer, store, test_ids, LIMITS,
        RUN_ROOT, "grpo_only", show_progress=SHOW_PROGRESS,
        training_result={"smoke": GRPO_ONLY_SMOKE, "full": GRPO_ONLY_FULL})
    manifest = g_only_full_manifest if GRPO_ONLY_FULL is not None else g_only_smoke_manifest
    export_adapter(model, tokenizer, VARIANT_DIRS["grpo_only"] / "adapter", manifest,
        extra={"test_metrics": GRPO_ONLY_METRICS, "smoke_verified": GRPO_ONLY_SMOKE,
               "full_training_result": GRPO_ONLY_FULL, "experiment_config": EXPERIMENT_CONFIG})
    save_json(VARIANT_DIRS["grpo_only"] / "memory_snapshots.json", [memory_snapshot("after_grpo_only")])
    display(pd.Series(GRPO_ONLY_METRICS, name="measured GRPO-only test").to_frame())
    display(HTML(render_traces_html(GRPO_ONLY_TRACES, label="GRPO only · test",
        count=INSPECT_EXAMPLES, offset=INSPECT_OFFSET, tokenizer=tokenizer)))
    model = release_trainer(g_only_final_trainer)
    g_only_smoke_trainer = g_only_full_trainer = g_only_final_trainer = None
else:
    print("GRPO-only evaluation not run; no performance is fabricated.")


GRPO-only evaluation not run; no performance is fabricated.


In [24]:
g_only_dir = VARIANT_DIRS["grpo_only"]
for name in ["rollout_metrics", "trainer_metrics", "evaluation_metrics", "evaluation_rewards"]:
    rows = read_rows(g_only_dir / (name + ".jsonl"))
    if rows:
        display(Markdown(f"### GRPO only · {name}"))
        display(pd.DataFrame(rows))
for name in ["rollout_metrics.png", "optimizer_metrics.png", "evaluation_rewards.png"]:
    path = g_only_dir / "figures" / name
    if path.is_file():
        display(Image(filename=str(path)))
g_only_training_traces = read_rows(g_only_dir / "training_trajectories.jsonl")
display(HTML(render_traces_html(g_only_training_traces, label="GRPO only · sampled training episodes",
    count=INSPECT_EXAMPLES, offset=INSPECT_OFFSET, tokenizer=tokenizer)))


<a id="comparison"></a>
## 14. Compare the four saved variants

Read the separate, already persisted evaluations. Columns represent vanilla, SFT, SFT + GRPO and GRPO only on identical test IDs. An unavailable variant has missing scores, not zeros. All RL return curves and trajectories were saved at their own stage boundaries, before this summary.


In [25]:
comparison = pd.DataFrame(save_comparison(RUN_ROOT))
display(comparison)
for variant, directory in VARIANT_DIRS.items():
    status_path = directory / "status.json"
    if status_path.is_file():
        display(pd.Series(json.loads(status_path.read_text()), name=variant).to_frame())
assert fingerprint_inputs(DATA_ROOT) == INPUT_SHA256, "The source dataset changed"
print("Separate policy artifacts:", RUN_ROOT / "variants")


,metric,Vanilla,SFT,SFT + GRPO,GRPO only
0,accuracy,None,None,None,None
1,mean_reward,None,None,None,None
2,mean_training_reward,None,None,None,None
3,retrieval_success_rate,None,None,None,None
4,valid_tool_call_rate,None,None,None,None
5,episodes_with_valid_calls_rate,None,None,None,None
6,episodes_all_calls_valid_rate,None,None,None,None
7,invalid_tool_call_rate,None,None,None,None
8,average_tool_calls,None,None,None,None
9,average_tool_calls_correct,None,None,None,None


Separate policy artifacts: /Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo/runs/kg_four_variants_seed41/variants


<a id="zip-heading"></a>
## 15. Package saved files for download

This cell only ZIPs files that earlier cells already saved. It can be run after an interrupted training/evaluation stage using the configured `RUN_ROOT`; partial diagnostics and previously completed variants are included. It requires no live model or trainer. Full optimizer checkpoints and Hub caches are excluded; retain checkpoints separately for resumption.


In [26]:
from kg_rlvr.training.artifacts import zip_run_artifacts
from IPython.display import FileLink
if MAKE_ZIP:
    zip_path = zip_run_artifacts(RUN_ROOT, WORK_ROOT / f"{RUN_NAME}_variants_and_diagnostics.zip")
    print("Download from the Kaggle output panel:", zip_path)
    display(FileLink(str(zip_path)))
else:
    print("ZIP creation disabled; per-variant files are already saved under", RUN_ROOT)


Download from the Kaggle output panel: /Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo/kg_four_variants_seed41_variants_and_diagnostics.zip


/Users/alirezaak/Documents/research/graph_rlvr/artifacts/grpo/kg_four_variants_seed41_variants_and_diagnostics.zip